![](https://site.unibo.it/rawmatcop-alliance/en/@@images/aa4c85c2-ecf6-48e4-a412-d3b49dc48ae7.png)

# Exercise 4 - SAM

In [ ]:
!uv pip install rasterio

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import spectral.io.envi as envi
from pystac_client import Client
import planetary_computer
from datetime import datetime, timedelta
from scipy.stats import linregress
from pathlib import Path
from glob import glob

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

ModuleNotFoundError: No module named 'rasterio'

## Loading Airborne Data

In [ ]:
LAB_5_DIR = Path().cwd() / "lab_5"
BASE_DIR = LAB_5_DIR / "data" / "you-shall-not-pass" / "Obrazy lotnicze"
print(BASE_DIR)
hdr_path = BASE_DIR / '221000_Odra_HS_Blok_A_008_VS_join_atm.hdr'

if hdr_path.exists():
    img = envi.open(hdr_path)
    print(f"Opened airborne cube: {hdr_path.name}")
    print(f"Shape: {img.shape}")
    
    # Since the cubes are large and memory overload is totally possible
    # only a subset is loaded
    cy, cx = img.nrows // 2, img.ncols // 2
    r_start, r_end = cy - 250, cy + 250
    c_start, c_end = cx - 250, cx + 250
    airborne_cube = img.read_subregion((r_start, r_end), (c_start, c_end))
    
    # Get wavelengths
    wavelengths = np.array([float(x) for x in img.metadata['wavelength']])
    print(f"Loaded subset shape: {airborne_cube.shape}")
else:
    print("Airborne data not found at expected path.")

ModuleNotFoundError: No module named 'envi'

## Load Spectral Library

Different land cover types (water, forest, green areas) are loaded as reference spectra for SAM.

In [ ]:
sig_dir = LAB_5_DIR / "spectral_signatures"
sig_files = list(sig_dir.glob('*.csv'))

spectra = {}
for f in sig_files:
    class_name = f.stem.rstrip('1234567890')
    df = pd.read_csv(f)
    if class_name not in spectra: spectra[class_name] = []
    spectra[class_name].append(df['value'].values)

# Calculate mean spectra for each class
reference_spectra = {k: np.nanmean(v, axis=0) for k, v in spectra.items()}
print(f"Reference classes: {list(reference_spectra.keys())}")

## Spectral Angle Mapper (SAM)

SAM calculates the spectral similarity between a reference spectrum and each pixel by computing the angle between them in a multidimensional spectral space. 

Smaller angles indicate higher similarity.

In this case SAM can distinguish between different types of water, which can show the contamination

It's calculated, because while the same pixel can appear brighter/darker based on conditions, but it will not change the spectral signature significantly, which can help identify different materials

In [ ]:
def compute_sam(hypercube, ref_spectrum):
    # ref_spectrum might need interpolation if band counts differ
    if len(ref_spectrum) != hypercube.shape[2]:
        ref_spectrum = np.interp(
            np.linspace(0, 1, hypercube.shape[2]), 
            np.linspace(0, 1, len(ref_spectrum)), 
            ref_spectrum
        )
    
    dot_product = np.sum(hypercube * ref_spectrum, axis=2)
    norm_cube = np.sqrt(np.sum(hypercube**2, axis=2))
    norm_ref = np.sqrt(np.sum(ref_spectrum**2))
    cos_theta = dot_product / (norm_cube * norm_ref + 1e-8)
    # clip values to arcos range to avoid numerical issues
    return np.arccos(np.clip(cos_theta, -1.0, 1.0))

# each class is applied the SAM
sam_maps = {k: compute_sam(airborne_cube, ref) for k, ref in reference_spectra.items()}

# minimal angle is used to classify pixels
class_names = list(sam_maps.keys())
stacked_angles = np.stack([sam_maps[k] for k in class_names], axis=-1)
classification = np.argmin(stacked_angles, axis=-1)

# plot the SAM classification map
plt.figure(figsize=(10, 8))
im = plt.imshow(classification, cmap='Set1')
cbar = plt.colorbar(im, ticks=range(len(class_names)))
cbar.ax.set_yticklabels(class_names)
plt.title('SAM Classification Map from airborne data')
plt.show()

## Fetch Sentinel-2 Data for aquisition date

**2025-07-17**

In [ ]:
# Bounding box approx from Polish CS2000 Z6 (Odra River)
bbox = [14.4, 52.8, 14.8, 53.2]
acq_date = "2025-07-17"

catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1", 
    modifier=planetary_computer.sign_inplace
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=acq_date,
    query={"eo:cloud_cover": {"lt": 20}}
)

items = list(search.items())
if items:
    best_item = items[0]
    print(f"Found Sentinel-2 item: {best_item.id}")
    
    # Access bands for RGB + NIR
    assets = best_item.assets
    b_blue = assets['B02'].href
    b_green = assets['B03'].href
    b_red = assets['B04'].href
    b_nir = assets['B08'].href
    
    with rasterio.open(b_red) as src:
        s2_red = src.read(1, out_shape=(500, 500)).astype('float32') / 10000.0
    print("Loaded resampled Sentinel-2 Red band.")
else:
    print("No Sentinel-2 image found for this date.")

## Calibrate Sentinel-2 using Airborne Data

To calibrate the satellite data, we use the airborne hyperspectral data as a high-resolution reference. We aggregate the airborne data to match the Sentinel-2 spectral response and spatial resolution, then compute a linear regression to find the gain and offset.

In [ ]:
# 1. Extract Red band from airborne cube (approx. 665 nm)
# The same method that was explained in exercise 3
red_idx = np.argmin(np.abs(wavelengths - 665))
airborne_red = airborne_cube[:, :, red_idx]

# 2. Resample airborne Red to match S2 (simple aggregation)
# (Assuming airborne is 1m and S2 is 10m, we aggregate 10x10 blocks)
h, w = airborne_red.shape
airborne_red_10m = airborne_red[:h//10*10, :w//10*10].reshape(h//10, 10, w//10, 10).mean(axis=(1, 3))

# 3. Crop S2 to match the airborne subset extent (approximate)
# this is done to have the same dimensions for both datasets
s2_subset = s2_red[:airborne_red_10m.shape[0], :airborne_red_10m.shape[1]]

# 4. Perform Linear Regression for Calibration
# please note that only non-nan values are used
mask = (airborne_red_10m > 0) & (s2_subset > 0)
slope, intercept, r_value, p_value, std_err = linregress(s2_subset[mask], airborne_red_10m[mask])

print(f"Calibration parameters: gain={slope:.4f}, offset={intercept:.4f}, R²={r_value**2:.4f}")

# 5. Apply Calibration
s2_calibrated = s2_red * slope + intercept

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(s2_red, cmap='gray')
plt.title('Original Sentinel-2 Red')
plt.subplot(1, 2, 2)
plt.imshow(s2_calibrated, cmap='gray')
plt.title('Calibrated Sentinel-2 Red')
plt.show()